# 03n - LoRA fine-tune of Qwen3.5-4B at the NATIVE resolution (thin wrapper)

Notebook 03 with `configs/finetune_qwen35_4b_native.yaml`: `max_pixels` 2196480 = 2,145 visual
tokens per page (the native 1240x1754 page, adopted by the 06 sweep), instead of 1,260. Everything
else is the 03 recipe (language-model-only LoRA r=16, 200 modules, 30,474,240 trainable parameters,
2 epochs, effective batch 8, seed 42). The 1260 config and the fold-0 adapter trained with it are
untouched. All logic is in `python -m shipdoc.train` / `python -m shipdoc.trainset`.

**Set `STAGE` in the parameters cell.** The default is `smoke`: the 20-step smoke-train at the
native resolution, the only thing to run first (peak VRAM at 2145 tokens is UNMEASURED; the
budget is the 03 one, 20 GiB bf16, never edited to pass). Stages: `smoke`, `fold0`, `fold1`,
`fold2` (cross-validation folds of `splits/folds.json`, train on the other two), `final` (all 400
train docs).

Run ids are `ft_native_<stage>_<sha7>_<precision>` (smoke: `ft_native_smoke_<sha7>_<precision>`) and
the config hash includes `max_pixels`, so a 1260 run folder or checkpoint is never reused. The
adapter manifest records `inference_keys.max_pixels`; the cell after the stage checks it is
2196480, and the OOF notebook refuses an adapter whose training resolution differs from its own.

Flow: parameters -> Drive + secrets -> clone at the pin -> unzip data -> **ESTIMATE (native VRAM,
hours and compute units per stage, printed before anything runs: read it)** -> install and GPU probe
-> prepare the training plan (CPU, the same `prepared_all.json` as 03: it holds no pixel sizes) ->
**SMOKE stage (always first; the notebook stops here if it fails)** -> the chosen `STAGE` (skipped
for `smoke`) -> resolution check of the manifest -> completion banner.

**Order of the stages (spec.md section 11 item 2).** `smoke` -> `fold0` (native) first -> its OOF
inference at the same resolution (05) -> `fold1`, `fold2`, `final`. All three folds and the final
adapter must be native: the 3-fold comparison of section 11 item 1 never mixes a 1260 adapter in.
Run the smoke ONCE before opening parallel tabs: they share the smoke folder on Drive. The
notebook does not enforce the order (no code gate); it prints a reminder.

Resumable exactly as 03 (checkpoints every 10 steps to `MyDrive/shipdoc-extract/runs/<run_id>/ckpt`;
Run all again resumes). No early stopping on held-out loss. Every number printed here is UNVERIFIED
until it has been measured on a GPU at this resolution.


In [ ]:
# Parameters. PINNED_SHA = the repo commit this run is pinned to (full 40-char SHA); the pin
# commit fills the placeholder (same two-commit pattern as 01_spike / 02_zeroshot500).
PINNED_SHA = "FILL_PINNED_SHA"
STAGE = "smoke"  # smoke | fold0 | fold1 | fold2 | final   (default: only the 20-step smoke)
STAGES = ["smoke", "fold0", "fold1", "fold2", "final"]
CONFIG = "configs/finetune_qwen35_4b_native.yaml"  # 03 config + max_pixels 2196480, seed 42
NATIVE_MAX_PIXELS = 2196480  # the estimate cell refuses a config that disagrees
FOLD_STAGES = ["fold0", "fold1", "fold2", "final"]
# W&B (curves only: train loss, grad norm, lr, held-out loss every 10 steps and at each epoch end;
# never images or values). ON by default for the fold0 / fold1 / fold2 / final stages, OFF for the
# smoke; override by hand. No WANDB_API_KEY secret or no network = W&B is skipped and training
# carries on: the run never depends on W&B. The held-out loss never stops training or picks a
# checkpoint (no early stopping, by design: it would leak the fold's held-out docs into G4).
USE_WANDB = STAGE in FOLD_STAGES
WANDB_PROJECT = "shipdoc-extract-debug"  # must be a PRIVATE project
# Stage order (a reminder, not a gate; spec.md section 11 item 2): fold0 NATIVE first, its OOF
# inference at the same resolution next, then fold1, fold2 and final.
AFTER_FOLD0 = (
    "fold0 (native) runs first and its OOF inference (05 at max_pixels 2196480) is read "
    "before fold1, fold2 and final; all of them must be native, never mix in a 1260 adapter."
)

if STAGE not in STAGES:
    raise ValueError(f"STAGE must be one of {STAGES}, got {STAGE!r}")
if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell (the pin commit fills it).")
if STAGE in ("fold1", "fold2", "final"):
    print("REMINDER:", AFTER_FOLD0)
SHA7 = PINNED_SHA[:7]
RUN_BASE = f"ft_native_{STAGE}_{SHA7}"  # precision appended once the GPU is known; never ft_*


## Account

No account check in this public build: Drive is mounted as whichever Google account you sign in
with, and the inputs must be in that account's `MyDrive/shipdoc-extract/` (see the repository
README, "Reproduce from images").


In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive

DRIVE_DIR = Path("/content/drive/MyDrive/shipdoc-extract")
REQUIRED_ZIPS = ["data.zip", "assignment.zip", "ocr_cache.zip"]
RUNS_DIR = DRIVE_DIR / "runs"

drive.mount("/content/drive")

print("Public build: any Google account; inputs are read from", DRIVE_DIR)

missing = [n for n in REQUIRED_ZIPS if not (DRIVE_DIR / n).is_file()]
if missing:
    raise FileNotFoundError(f"Missing in MyDrive/shipdoc-extract/: {', '.join(missing)}")
print("Drive inputs present:", ", ".join(REQUIRED_ZIPS))
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("runs folder on Drive:", RUNS_DIR)


In [ ]:
import os

from google.colab import userdata


def get_secret(name: str, required: bool) -> str | None:
    """Read a Colab secret without ever printing it."""
    try:
        value = userdata.get(name)
    except Exception:  # SecretNotFoundError / NotebookAccessError: both mean "not usable"
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret {name} is missing or notebook access is off "
            "(left sidebar -> key icon -> add it and enable Notebook access)."
        )
    return value or None


WANDB_API_KEY = get_secret("WANDB_API_KEY", required=False)
HF_TOKEN = get_secret("HF_TOKEN", required=False)
print("WANDB_API_KEY:", "set" if WANDB_API_KEY else "not set",
      "| HF_TOKEN:", "set" if HF_TOKEN else "not set (optional)")


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "FILL_PUBLIC_REPO_URL"
REPO = Path("/content/shipdoc-extract")
if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell to the commit this run is pinned to.")
if "FILL" in REPO_URL:
    raise ValueError("REPO_URL is a placeholder: the public repository is not published yet.")


def git(*args: str) -> str:
    """Run git in the repo dir (or /content). The repository is public: no credentials at all."""
    cwd = REPO if (REPO / ".git").is_dir() else "/content"
    res = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)
    if res.returncode:
        raise RuntimeError(f"git {args[0]} failed (exit {res.returncode}): {res.stderr[-800:]}")
    return res.stdout.strip()


if (REPO / ".git").is_dir():
    git("fetch", "origin")
else:
    git("clone", REPO_URL, str(REPO))
git("checkout", PINNED_SHA)
head = git("rev-parse", "HEAD")
pinned_full = git("rev-parse", f"{PINNED_SHA}^{{commit}}")
assert head == pinned_full, f"HEAD {head} != PINNED_SHA {pinned_full}"
print("HEAD =", head)


In [ ]:
import hashlib
import shutil
import zipfile

LOCAL_ZIPS = Path("/content/zips")
LOCAL_ZIPS.mkdir(exist_ok=True)
# data.zip holds data/, assignment.zip holds assignment/, ocr_cache.zip holds ocr_cache/.
TARGETS = {"data.zip": REPO, "assignment.zip": REPO, "ocr_cache.zip": Path("/content")}
for name, dest in TARGETS.items():
    local = LOCAL_ZIPS / name
    if not local.is_file() or local.stat().st_size != (DRIVE_DIR / name).stat().st_size:
        shutil.copyfile(DRIVE_DIR / name, local)  # local disk: Drive reads are slow
    with zipfile.ZipFile(local) as zf:
        top = zf.namelist()[0].split("/")[0]
        if not (dest / top).is_dir():
            zf.extractall(dest)
    print(f"{name}: unzipped to {dest / top}")

# Verify the OCR cache against its own SHA256SUMS (paths are relative to ocr_cache/).
OCR_CACHE = Path("/content/ocr_cache")
bad = []
sums = (OCR_CACHE / "SHA256SUMS").read_text(encoding="utf-8").splitlines()
for line in sums:
    digest, _, rel = line.partition("  ")
    f = OCR_CACHE / rel
    if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != digest:
        bad.append(rel)
if bad:
    raise RuntimeError(f"ocr_cache SHA256SUMS: {len(bad)} bad/missing files, e.g. {bad[:5]}")
print(f"ocr_cache OK: {len(sums)} files verified")


In [ ]:
# ESTIMATE of VRAM, GPU hours and compute units (CU) at the NATIVE resolution, printed BEFORE
# anything runs. No confirmation prompt (Run all must not block): interrupt it yourself if it is
# too high. Source: scripts/finetune_native_estimate.py, a wrapper over finetune_estimate.py that
# scales the MEASURED 1260-token fold-0 pace (8.87 s/page) and peak VRAM (11.4 GiB) by stated,
# ASSUMED linear factors. UNVERIFIED: the smoke below measures the real peak and s/step.
import importlib.util
import subprocess
import sys

import yaml

_spec = importlib.util.spec_from_file_location(
    "finetune_native_estimate", REPO / "scripts" / "finetune_native_estimate.py"
)
fne = importlib.util.module_from_spec(_spec)
sys.modules["finetune_native_estimate"] = fne  # dataclasses resolve their module via sys.modules
_spec.loader.exec_module(fne)
fe = fne.fe

cfg = yaml.safe_load((REPO / CONFIG).read_text(encoding="utf-8"))
assert cfg["max_pixels"] == NATIVE_MAX_PIXELS, (
    f"{CONFIG} has max_pixels {cfg['max_pixels']}, this notebook is the native one "
    f"({NATIVE_MAX_PIXELS}): use notebook 03 for the 1260 config."
)
assert cfg["expected_visual_tokens"] == fne.TOKENS_NATIVE, cfg["expected_visual_tokens"]
EPOCHS, ACCUM = cfg["epochs"], cfg["grad_accum"]


def read_gpu():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=30, check=False,
        ).stdout.strip().splitlines()
    except (OSError, subprocess.TimeoutExpired):
        return None, None
    if not out:
        return None, None
    name, _, cap = out[0].partition(",")
    try:
        major = int(cap.strip().split(".")[0])
    except ValueError:
        major = None
    return name.strip(), major


GPU_NAME, GPU_MAJOR = read_gpu()
# Same rule as shipdoc.train.resolve_precision (the install cell asserts they agree).
PREVIEW_PRECISION = "fp32" if GPU_MAJOR is None else ("bf16" if GPU_MAJOR >= 8 else "fp16")
KIND = next((k for k in ("L4", "T4") if GPU_NAME and k in GPU_NAME.split()), None)
pages = fe.fold_pages(REPO)
print(f"Runtime GPU (nvidia-smi): {GPU_NAME} | "
      f"precision this run will use: {PREVIEW_PRECISION}")
print(f"STAGE = {STAGE!r}; {EPOCHS} epochs, effective batch {ACCUM} pages, max_pixels "
      f"{cfg['max_pixels']} ({cfg['expected_visual_tokens']} visual tokens per page); the smoke "
      "stage always runs first.")
if KIND != "L4":
    print("WARNING: native training is planned on an L4 (bf16). On a T4 (fp16, 13.5 GiB "
          "budget) the high VRAM estimate below is inside 15% of it: expect the smoke to fail.")
print()
for line in fne.native_lines(pages, EPOCHS, ACCUM, cfg["eval_every"], cfg["eval_pages"]):
    print(line)
print("\nOOF and dev INFERENCE (T4 fp16) is not part of this notebook. " + AFTER_FOLD0)


In [ ]:
import collections
import json
import subprocess
import time
from datetime import UTC, datetime

PY = str(REPO / ".venv" / "bin" / "python")
ENV = {
    **os.environ,
    "SHIPDOC_PROFILE": "colab",
    "HF_HOME": "/content/hf_cache",
    "SHIPDOC_RUNS_DIR": str(RUNS_DIR),
    "SHIPDOC_DATA_DIR": str(REPO / "data"),
    "SHIPDOC_ASSIGNMENT_DIR": str(REPO / "assignment"),
    "SHIPDOC_OCR_CACHE": "/content/ocr_cache",
    "SHIPDOC_TMP_DIR": "/content/tmp",
    "UV_CACHE_DIR": "/content/uv_cache",
    "PYTORCH_ALLOC_CONF": "expandable_segments:True",
}
if USE_WANDB and WANDB_API_KEY:
    ENV.update(SHIPDOC_WANDB="1", WANDB_API_KEY=WANDB_API_KEY, WANDB_PROJECT=WANDB_PROJECT)
    print(f"W&B: ON, private project {WANDB_PROJECT} (curves only; never stops the run)")
elif USE_WANDB:
    print("W&B: OFF - USE_WANDB is on but the Colab secret WANDB_API_KEY is not set. Training is "
          "unaffected; add the secret and re-run to get the curves.")
else:
    print("W&B: OFF (USE_WANDB is False for this stage)")
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
Path(ENV["SHIPDOC_TMP_DIR"]).mkdir(exist_ok=True)


def run_stream(
    cmd: list[str], tail: int = 40, env: dict | None = None
) -> tuple[int, list[str]]:
    """Run a command, echo its output live, return (exit code, last `tail` lines)."""
    last: collections.deque[str] = collections.deque(maxlen=tail)
    with subprocess.Popen(
        cmd, cwd=REPO, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as proc:
        for line in proc.stdout:
            print(line, end="")
            last.append(line.rstrip("\n"))
    return proc.returncode, list(last)


def save_json(path: Path, obj) -> None:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=1))
    tmp.replace(path)  # atomic: a disconnect never leaves a half-written file


def run_session(cmd: list[str], run_dir: Path) -> tuple[int, list[str]]:
    """run_stream + one wall-clock entry per call in <run_dir>/sessions.json."""
    run_dir.mkdir(parents=True, exist_ok=True)
    path = run_dir / "sessions.json"
    sessions = json.loads(path.read_text()) if path.is_file() else []
    session = {"start": datetime.now(UTC).isoformat(), "seconds": None, "exit_code": None}
    sessions.append(session)
    save_json(path, sessions)
    t0 = time.time()
    try:
        rc, tail = run_stream(cmd)
    finally:
        session["seconds"] = round(time.time() - t0, 1)
        save_json(path, sessions)
    session["exit_code"] = rc
    save_json(path, sessions)
    return rc, tail


if subprocess.run(["uv", "--version"], capture_output=True, check=False).returncode:
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
# peft==0.21.2 comes from the locked `train` dependency group (uv.lock), like everything else.
rc, _ = run_stream(
    ["uv", "sync", "--frozen", "--group", "vlm", "--group", "train", "--python", "3.11"]
)
assert rc == 0, f"uv sync failed (exit {rc})"

subprocess.run(["nvidia-smi"], check=False)
PROBE = (
    "import json, importlib.metadata as m, torch;"
    "from shipdoc.train import resolve_precision;"
    "p = torch.cuda.get_device_properties(0);"
    "print(json.dumps({'torch': torch.__version__, 'cuda': torch.cuda.is_available(),"
    " 'gpu': torch.cuda.get_device_name(0), 'major': p.major,"
    " 'vram_gib': round(p.total_memory / 2**30, 1),"
    " 'precision': resolve_precision('auto', p.major),"
    " **{k: m.version(k) for k in ('transformers', 'peft', 'accelerate')}}))"
)
info = json.loads(subprocess.run([PY, "-c", PROBE], cwd=REPO, env=ENV, capture_output=True,
                                 text=True, check=True).stdout.strip().splitlines()[-1])
print(info)
assert info["cuda"], "No CUDA device: Runtime -> Change runtime type -> L4 (recommended) or T4."
PRECISION = info["precision"]
assert PRECISION == PREVIEW_PRECISION, (
    f"estimate cell predicted {PREVIEW_PRECISION}, torch says {PRECISION} (GPU {info['gpu']})")
print(f"GPU {info['gpu']} ({info['vram_gib']} GiB): training precision = {PRECISION} "
      f"({'bf16, no scaler' if PRECISION == 'bf16' else 'fp16 + GradScaler + fp32 LoRA weights'})")
RUN_ID = f"{RUN_BASE}_{PRECISION}"
SMOKE_RUN_ID = f"ft_native_smoke_{SHA7}_{PRECISION}"  # never the 1260 ft_smoke_* folder
RUN_DIR, SMOKE_DIR = RUNS_DIR / RUN_ID, RUNS_DIR / SMOKE_RUN_ID
print("run_id:", RUN_ID, "| smoke run_id:", SMOKE_RUN_ID)
print("Drive output folder:", RUN_DIR if STAGE != "smoke" else SMOKE_DIR)


In [ ]:
# Training plan for all 500 train+dev docs (CPU, a few minutes, cached on Drive): the page of
# every gold row from the OCR locator, the header-only / excluded documents, and the occlusion
# candidates. Counts only are printed; the audit log (doc id, row, page, decision; no values)
# is written next to the cache. Both fold and final stages read this one file.
PREPARED = RUNS_DIR / "prepared_all.json"
if PREPARED.is_file():
    print(f"SKIP prepare: {PREPARED} exists ({PREPARED.stat().st_size / 1e6:.1f} MB)")
else:
    rc, _ = run_stream([PY, "-m", "shipdoc.trainset", "--stage", "final", "--all",
                        "--out", str(PREPARED)])
    assert rc == 0, f"trainset preparation failed (exit {rc})"
    print("audit log:", PREPARED.parent / "audit.jsonl")


In [ ]:
# MANDATORY smoke stage (the training path is unverified on a GPU): preflight the longest page,
# then 20 optimizer steps (accum 2), then check (i) the loss decreased (mean of the last 5 steps
# below 0.95 x the first 5), (ii) peak VRAM under the configured budget, (iii) 200 LoRA modules /
# 30,474,240 trainable parameters, (iv) no NaN / inf, plus a checkpoint write+reload on Drive.
# Any failure raises HERE: nothing after this cell runs. A passed smoke is not repeated.
SMOKE_STATUS = SMOKE_DIR / "smoke_status.json"


def read_status(path: Path) -> dict:
    try:
        return json.loads(path.read_text())
    except (OSError, ValueError):
        return {}


status = read_status(SMOKE_STATUS)
if status.get("state") == "passed":
    SMOKE_PASSED = True
    print(f"SKIP smoke: already passed ({SMOKE_RUN_ID})")
else:
    rc, tail = run_session(
        [PY, "-m", "shipdoc.train", "--config", CONFIG, "--stage", "smoke",
         "--run-dir", str(SMOKE_DIR)], SMOKE_DIR)
    status = read_status(SMOKE_STATUS)
    SMOKE_PASSED = rc == 0 and status.get("state") == "passed"
    for c in status.get("checks", []):
        print(f"{c['status'].upper():<5} {c['name']:<20} {c['detail']}")
if not SMOKE_PASSED:
    raise RuntimeError(
        "SMOKE GATE FAILED: the stage is NOT started. GG: report back with "
        f"{SMOKE_STATUS} and the per-check table printed above (no document values in them). "
        "Do not edit thresholds to make it pass."
    )
print(f"SMOKE PASSED ({status.get('precision')}, peak {status.get('peak_vram_gib', 0):.1f} GiB)."
      " Measured s/step is in metrics.jsonl: rerun scripts/finetune_estimate.py with it.")


In [ ]:
# The chosen stage, RESUMABLE: checkpoints every 10 steps go to <run>/ckpt on Drive and Run all
# again continues from the newest one (bit-exact resume). STAGE = "smoke" stops after the smoke.
import re

if not SMOKE_PASSED:
    raise RuntimeError("The smoke gate has not passed: the stage is NOT started.")
OOM_RE = re.compile(r"out of memory|OutOfMemoryError", re.IGNORECASE)
if STAGE in ("fold1", "fold2", "final"):
    print("REMINDER:", AFTER_FOLD0)
if STAGE == "smoke":
    print('STAGE = "smoke": nothing more to run. To train, set STAGE to fold0 (first) in the '
          'parameters cell and Run all (the smoke is skipped, it already passed). ' + AFTER_FOLD0)
else:
    train_status = read_status(RUN_DIR / "train_status.json")
    if train_status.get("state") == "complete":
        print(f"SKIP {STAGE}: already complete ({RUN_DIR})")
    else:
        print(f"=== {RUN_ID}: training (resumes from the newest checkpoint if any) ===")
        rc, tail = run_session(
            [PY, "-m", "shipdoc.train", "--config", CONFIG, "--stage", STAGE,
             "--run-dir", str(RUN_DIR)], RUN_DIR)
        if rc != 0:
            oom = OOM_RE.search("\n".join(tail))
            raise RuntimeError(
                f"{STAGE} stopped (exit {rc}{', CUDA out of memory' if oom else ''}); last "
                f"output: {tail[-3:]}. Everything checkpointed so far is on Drive: Run all again "
                "resumes. If it stops again at the same step, report back."
            )


In [ ]:
# Resolution record of the finished stage: <run>/final/manifest.json must carry the training
# resolution (inference_keys.max_pixels) and it must be the native one. Notebook 05 runs the same
# check (shipdoc.resmatch, stdlib only) and refuses an adapter trained at another resolution.
import importlib.util
import sys

_spec = importlib.util.spec_from_file_location("resmatch", REPO / "src" / "shipdoc" / "resmatch.py")
rm = importlib.util.module_from_spec(_spec)
sys.modules["resmatch"] = rm
_spec.loader.exec_module(rm)
if STAGE == "smoke":
    print("STAGE = smoke: no adapter manifest to check.")
else:
    manifest_path = RUN_DIR / "final" / "manifest.json"
    if not manifest_path.is_file():
        raise RuntimeError(f"{manifest_path} is missing: the stage did not finish.")
    manifest = json.loads(manifest_path.read_text())
    ok, why = rm.training_resolution_matches(manifest, {"max_pixels": NATIVE_MAX_PIXELS})
    print(("RESOLUTION OK: " if ok else "RESOLUTION MISMATCH: ") + why)
    if not ok:
        raise RuntimeError("The adapter manifest does not record the native resolution: " + why)


In [ ]:
# Completion banner. Reads only files on Drive: also runnable alone after a resume.
import importlib.util
import sys

# metrics.jsonl is read through shipdoc.metricslog (stdlib only): one line per optimizer step,
# steps 1..n once each. A file from a build before the fix can hold a dead session's lines too
# (the first smoke printed 29 / 20); the reader then keeps the last complete run and says so.
_spec = importlib.util.spec_from_file_location(
    "metricslog", REPO / "src" / "shipdoc" / "metricslog.py"
)
ml = importlib.util.module_from_spec(_spec)
sys.modules["metricslog"] = ml  # dataclasses resolve their module through sys.modules
_spec.loader.exec_module(ml)
DIR = SMOKE_DIR if STAGE == "smoke" else RUN_DIR
metrics_path = DIR / "metrics.jsonl"
STATUS_NAME = "smoke_status.json" if STAGE == "smoke" else "train_status.json"
status = read_status(DIR / STATUS_NAME)
try:
    metrics = ml.read_metrics(metrics_path, status.get("total_steps"), repair=True)
except ml.MetricsAnomaly as exc:  # e.g. a gap: show nothing rather than a wrong curve
    print("WARNING: metrics.jsonl unreadable:", exc)
    metrics = ml.MetricsRead([], 0, [str(exc)])
rows = metrics.rows
SESSIONS = DIR / "sessions.json"
sessions = json.loads(SESSIONS.read_text()) if SESSIONS.is_file() else []
secs = sum(s["seconds"] for s in sessions if s.get("seconds") is not None)
unfinished = sum(s.get("seconds") is None for s in sessions)
losses = [r["loss"] for r in rows]
k = 5
bar = "=" * 78
print(bar)
print(f"FINE-TUNE {STAGE.upper()} {status.get('state', 'unknown').upper()}: {DIR.name}")
print(bar)
print(f"GPU / precision : {info['gpu']} / {PRECISION}")
print(f"optimizer steps : {len(rows)} / {status.get('total_steps', '?')}"
      f"  ({status.get('n_train_pages', '?')} pages in the stage's pool)")
if metrics.repaired:
    print(f"WARNING         : metrics.jsonl had {metrics.n_lines} lines for {len(rows)} steps "
          f"({'; '.join(metrics.problems)}); showing the last complete run of steps")
if len(losses) >= 2 * k:
    first, last = sum(losses[:k]) / k, sum(losses[-k:]) / k
    print(f"loss curve      : first {k} steps {first:.4f} -> last {k} steps {last:.4f} "
          f"({100 * (last - first) / first:+.1f}%), min {min(losses):.4f}")
elif losses:
    print(f"loss curve      : {len(losses)} steps, first {losses[0]:.4f}, last {losses[-1]:.4f}")
evals = [(r["step"], r["eval_loss"]) for r in rows if r.get("eval_loss") is not None]
if evals:
    print("held-out loss   : " + ", ".join(f"step {s}: {v:.4f}" for s, v in evals)
          + "  (information only; no early stopping)")
ep_evals = [(r["epoch_done"], r["eval_loss_epoch"]) for r in rows if "eval_loss_epoch" in r]
if ep_evals:
    print("held-out / epoch: " + ", ".join(f"epoch {e}: {v:.4f}" for e, v in ep_evals)
          + "  (24 fixed held-out pages; curves only, never used to stop or to pick a checkpoint)")
if rows:
    print(f"s/step (mean)   : {sum(r['seconds'] for r in rows) / len(rows):.1f}   "
          f"scaler skips: {sum(bool(r.get('scaler_skipped')) for r in rows)}")
print(f"peak VRAM       : {status.get('peak_vram_gib', 0):.2f} GiB")
print(f"wall-clock      : {secs / 3600:.2f} h over {len(sessions) - unfinished} finished "
      f"session(s)" + (f" (+ {unfinished} without a recorded end: killed or disconnected, time "
                       "not counted)" if unfinished else ""))
print(f"Drive folder    : {DIR}")
print("GG: download these files from that folder:")
names = ["smoke_status.json", "metrics.jsonl", "sessions.json"] if STAGE == "smoke" else [
    "final/adapter.pt", "final/manifest.json", "train_status.json", "metrics.jsonl",
    "sessions.json"]
for name in names:
    path = DIR / name
    size = f"{path.stat().st_size / 1e6:8.2f} MB" if path.is_file() else "MISSING"
    print(f"  {name:<20}{size}")
if STAGE != "smoke":
    peft_dir = DIR / "final" / "peft"
    for path in sorted(peft_dir.glob("*")) if peft_dir.is_dir() else []:
        print(f"  final/peft/{path.name:<14}{path.stat().st_size / 1e6:8.2f} MB")
print(f"also useful (not needed): {RUNS_DIR / 'audit.jsonl'}")
if STAGE == "fold0":
    print(bar)
    print("NEXT (fold0): the adapter folder is " + str(DIR / "final"))
    print("(adapter.pt, manifest.json, peft/). Run the OOF inference of fold 0's held-out pages")
    print("(230) on a T4 and compare it")
    print("with zero-shot at max_pixels 2196480 (and the 1260 fold-0 result) BEFORE anything else.")
    print("REMINDER:", AFTER_FOLD0)
    print(bar)
print(f"DONE {RUN_ID if STAGE != 'smoke' else SMOKE_RUN_ID} state={status.get('state')}")
